
# Vehicle Performance RAG Chatbot  
### Project 2 – Modular Local RAG System (Mistral-7B + CUDA + LangChain + Gradio)

**Purpose:**  
To build a local, GPU-accelerated RAG-based analytics assistant that connects to the SQL database from *Project 1* and supports semantic querying using LangChain, Mistral-7B, and ChromaDB.


In [ ]:
# ✅ Cell 0 — Clean modern stack (no conflicts)
!pip install -qU langchain-core langchain-community langchain-text-splitters \
chromadb sentence-transformers transformers accelerate gradio langgraph langchain

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA runtime version:", torch.version.cuda)
else:
    print("⚠️ CUDA not detected — check driver installation or restart kernel.")

In [ ]:
# Cell 1: Environment Setup & Imports (Final – Modern LangChain 2026)
# -------------------------------------------------------------------
# Purpose:
# - Import all required modules (updated for modular LangChain v0.3+)
# - Detect GPU availability
# - Initialize project paths
# - Ensure compatibility with modern RAG APIs

import os
import torch
import yaml
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

# Modern LangChain modules
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gradio as gr


# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Paths
PROJECT_ROOT = Path(r"C:\VSCode\vscode_work\resume_projects\DS-AI-VEH-Portfolio\project2_vehicle_performance_rag")
DATA_DIR = PROJECT_ROOT / "data"
CHROMA_DIR = DATA_DIR / "chroma_db"
DOCS_DIR = DATA_DIR / "docs"
CONFIG_PATH = PROJECT_ROOT / "config" / "db_config.yaml"

CHROMA_DIR.mkdir(parents=True, exist_ok=True)


print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📄 Docs path: {DOCS_DIR}")
print(f"💾 Chroma path: {CHROMA_DIR}")
print("✅ Environment initialized successfully.")


In [ ]:

# Cell 2: Load Configuration
# --------------------------
# Purpose:
# - Read database credentials from YAML config file

with open(CONFIG_PATH, "r") as f:
    db_cfg = yaml.safe_load(f)

MYSQL_USER = db_cfg.get("user", "root")
MYSQL_PASSWORD = db_cfg.get("password", "mypassword!")
MYSQL_HOST = db_cfg.get("host", "localhost")
MYSQL_PORT = db_cfg.get("port", 3306)
MYSQL_DB = db_cfg.get("database", "telemetry_analytics_db")

print("Loaded database configuration successfully.")


In [ ]:

# Cell 3: Database Connection Utility
# -----------------------------------
# Purpose:
# - Create reusable SQLAlchemy engine for live analytics queries

def get_db_engine():
    return create_engine(f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}")

engine = get_db_engine()
print(f"Connected to MySQL database: {MYSQL_DB}")


In [ ]:

# Cell 4: Load Reference Documents
# --------------------------------
# Purpose:
# - Load markdown files containing analytics definitions, architecture, data models, etc.

from langchain_core.documents import Document

docs = []
for md_file in DOCS_DIR.glob("*.md"):
    with open(md_file, "r", encoding="utf-8") as f:
        text = f.read()
        docs.append(Document(page_content=text, metadata={"source": md_file.name}))

print(f"Loaded {len(docs)} reference documents.")


In [ ]:

# Cell 5: Vectorization & ChromaDB Store
# --------------------------------------
# Purpose:
# - Split text and create embeddings
# - Store them in a local ChromaDB instance

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
split_docs = splitter.split_documents(docs)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(split_docs, embedding_model, persist_directory=str(CHROMA_DIR))
vectorstore.persist()

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("ChromaDB vector store created and persisted.")


In [ ]:

# Cell 6: LLM Loader (Mistral-7B + CUDA)
# --------------------------------------
# Purpose:
# - Load local Mistral model for causal text generation

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if device == "cuda" else torch.float32, device_map="auto")

generation_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,
)

llm = HuggingFacePipeline(pipeline=generation_pipe)
print("Mistral-7B model loaded successfully.")


In [ ]:
llm = HuggingFacePipeline(pipeline=generation_pipe)
llm.invoke("Explain fuel efficiency and in MT and AMT transmission.")

In [ ]:
# Cell 7: RAG Pipeline Construction (FINAL – Stateless, Stable)
# ------------------------------------------------------------
# Purpose:
# - Build stable RAG pipeline for local HuggingFace models
# - Avoid memory-related list hashing errors
# - Production-safe for offline deployment

from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

# Format retrieved documents
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert vehicle performance analytics assistant."),
    ("human", "Context:\n{context}\n\nQuestion:\n{input}")
])

# Stateless RAG chain (STABLE)
rag_chain = (
    RunnableParallel(
        context=retriever | format_docs,
        input=RunnablePassthrough()
    )
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Stateless RAG pipeline initialized (stable for local LLMs).")


In [ ]:
# Cell 8: Chat Agent Assembly (Gradio-compatible)
# -----------------------------------------------
# Purpose:
# - Match Gradio ChatInterface signature
# - Keep RAG chain stateless and stable

def chat_with_rag(message, history):
    try:
        response = rag_chain.invoke(message)
        return response
    except Exception as e:
        return f"Error: {str(e)}"


In [ ]:

# Cell 9: Gradio Interface
# ------------------------
# Purpose:
# - Build chat interface for the RAG system

def chat_interface(query, history=[]):
    response = chat_with_rag(query)
    history.append((query, response))
    return history, history

chat_ui = gr.ChatInterface(fn=chat_with_rag, title="Vehicle Performance RAG Chatbot", description="Conversational analytics assistant for telemetry-based performance insights.")

chat_ui.launch(debug=True)


In [ ]:

# Cell 10: Example Queries
# ------------------------
# Purpose:
# - Demonstrate sample queries

examples = [
    "Summarize the main analytics modules in this project.",
    "Explain how frequency-domain analytics works in vehicle performance context.",
    "What insights can be derived from actuation statistics?",
    "Describe the architecture flow of this analytics pipeline."
]

for q in examples:
    print(f"Q: {q}\nA: {chat_with_rag(q)}\n{'-'*80}")
